# Module 3.4 - Semantic Caching

**Reference:** [`04-semantic-caching.md`](./04-semantic-caching.md)

## What you'll do in this notebook

- Show why an exact-match dict-cache barely helps for natural-language queries.
- Build a semantic cache using the same embedding machinery as RAG.
- Tune the similarity threshold to balance hit rate against false positives.
- Wrap a baseline RAG chatbot with the cache and measure the savings.

## Setup

In [ ]:
import os
import time
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBED_MODEL = "text-embedding-3-small"
print(f"OK: client ready, model = {MODEL}")

## Exercise 1 - Exact-match caching fails

Two users ask the same question phrased differently. A dict-cache keyed on the raw string has no idea they mean the same thing.

In [ ]:
exact_cache: dict[str, str] = {}

def expensive_answer(query: str) -> str:
    """Stand-in for a full RAG + LLM call."""
    time.sleep(0.2)  # pretend this took real work
    return f"(generated answer for: {query})"

def exact_chat(query: str) -> tuple[str, bool]:
    """Returns (answer, was_cached)."""
    if query in exact_cache:
        return exact_cache[query], True
    ans = expensive_answer(query)
    exact_cache[query] = ans
    return ans, False

queries = [
    "How do I deploy my app?",
    "How to deploy application?",
    "Deployment instructions?",
    "Steps to deploy",
]

for q in queries:
    ans, cached = exact_chat(q)
    print(f"[{'HIT ' if cached else 'miss'}] {q}")

You should see four misses. Four real calls for four paraphrases of the same question.

## Exercise 2 - A semantic cache from scratch

Instead of storing queries in a dict, store their embeddings. On a new query, embed it and compare against the cached embeddings with cosine similarity. If any stored embedding is above the threshold, return the cached answer.

In [ ]:
def embed_one(text: str) -> np.ndarray:
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.asarray(resp.data[0].embedding)

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

class SemanticCache:
    def __init__(self, similarity_threshold: float = 0.95, ttl_hours: int = 24):
        self.similarity_threshold = similarity_threshold
        self.ttl = timedelta(hours=ttl_hours)
        self._entries: list[dict] = []  # each: {query, embedding, answer, stored_at}

    def find(self, query: str) -> str | None:
        # TODO:
        # 1. If self._entries is empty, return None.
        # 2. Embed query.
        # 3. Compute cosine similarity against every stored embedding.
        # 4. Pick the best match. If similarity >= threshold and (now - stored_at) < ttl,
        #    return the stored answer. Otherwise return None.
        raise NotImplementedError

    def store(self, query: str, answer: str) -> None:
        self._entries.append({
            "query": query,
            "embedding": embed_one(query),
            "answer": answer,
            "stored_at": datetime.now(),
        })

cache = SemanticCache(similarity_threshold=0.93)

def semantic_chat(query: str) -> tuple[str, bool]:
    hit = cache.find(query)
    if hit is not None:
        return hit, True
    ans = expensive_answer(query)
    cache.store(query, ans)
    return ans, False

for q in queries:
    ans, cached = semantic_chat(q)
    print(f"[{'HIT ' if cached else 'miss'}] {q}")

The first query misses; the next three should all hit because they're paraphrases of the same question.

## Exercise 3 - Tuning the threshold

Too high: the cache never hits. Too low: unrelated questions match and you return a wrong answer. Find a good middle ground.

In [ ]:
pairs = [
    ("How do I deploy my app?", "How to deploy application?", True),
    ("How do I deploy my app?", "Steps to deploy", True),
    ("How do I deploy my app?", "What is Python?", False),
    ("How do I deploy my app?", "How do I roll back a deployment?", False),
    ("How do I catch errors in Python?", "Python exception handling", True),
]

print(f"{'similarity':>10}  expected  query_a   |   query_b")
for a, b, expected in pairs:
    sim = cosine(embed_one(a), embed_one(b))
    print(f"{sim:10.3f}  {str(expected):>8}   {a!r}   |   {b!r}")

**Reflect:** looking at the similarities above, pick a threshold that correctly accepts the `True` pairs and rejects the `False` pairs. Would that threshold change if your docs corpus covered a single narrow domain instead of a broad one?

## Exercise 4 - Wrap a RAG chatbot with the cache

In practice, the cache sits in front of the full RAG pipeline. On a hit, you skip *both* the retrieval and the LLM generation.

In [ ]:
class CachedRAG:
    def __init__(self, rag_fn, similarity_threshold: float = 0.93):
        self.rag_fn = rag_fn  # takes a str, returns a str
        self.cache = SemanticCache(similarity_threshold=similarity_threshold)

    def chat(self, query: str) -> tuple[str, bool]:
        # TODO:
        # 1. Ask the cache.
        # 2. On hit, return (answer, True).
        # 3. On miss, call self.rag_fn(query), store the result, return (answer, False).
        raise NotImplementedError


# Demo with a fake RAG that just sleeps
cached = CachedRAG(expensive_answer)

start = time.time()
for q in queries:
    ans, cached_hit = cached.chat(q)
    print(f"[{'HIT ' if cached_hit else 'miss'}] {q}")
print(f"\nTotal elapsed: {time.time() - start:.2f}s")
print("Without the cache, four misses would take ~0.8s just from the stubbed delay;")
print("in real life, you're saving several seconds per hit plus the LLM cost.")

## What's next

You now have five ways to improve retrieval (better queries, reranking, hybrid, knowledge graphs, caching). In `05-evaluating-and-optimising-retrieval.ipynb` we stop guessing which one helps and measure it, with Precision@K, Recall@K, and MRR on a hand-built test set.